In [0]:
%pip install openpyxl
%pip install pymc arviz

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Load data ---
file_path = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/EXH_ModelData.xlsx"

df = pd.read_excel(file_path, sheet_name=0)

# Sort chronologically
df = df.sort_values("week_date").reset_index(drop=True)

print(f"Shape : {df.shape}")
print(f"Range : {df['week_date'].min().date()} → {df['week_date'].max().date()}")
print(f"Weeks : {len(df)}")
print(f"Missing values: {df.isna().sum().sum()}")

In [0]:
# --- Column roles ---
date_col     = "week_date"
target_col   = "total_weekly_attendance"
control_cols = df.columns[3:45].tolist()   # cols 4-45 (1-indexed)

print(f"Date     : {date_col}")
print(f"Target   : {target_col}")
print(f"Controls : {len(control_cols)} columns")

# Sanity checks
assert target_col not in control_cols
assert date_col not in control_cols
week_diffs = df[date_col].diff().dropna().dt.days.value_counts()
print(f"\nWeek spacing: {week_diffs.to_dict()}")  # expect {7: 164}

In [0]:
# --- Encode the 3 non-numeric controls ---

# Binary flags: Y/N → 1/0
df["stat_holiday_in_week"] = (df["stat_holiday_in_week"] == "Y").astype(int)
df["is_long_weekend"]      = (df["is_long_weekend"] == "Y").astype(int)

# Season: one-hot, drop 'winter' as reference level
season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=False).astype(int)
season_dummies = season_dummies.drop(columns=["season_winter"])
df = pd.concat([df.drop(columns=["season"]), season_dummies], axis=1)

# Update control_cols: remove 'season', add the 4 new dummies
control_cols = [c for c in control_cols if c != "season"] + season_dummies.columns.tolist()

# Verify all numeric
non_numeric = df[control_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Final control count : {len(control_cols)}")
print(f"Non-numeric remaining: {non_numeric if non_numeric else 'None ✓'}")
print(f"Season dummies      : {season_dummies.columns.tolist()}")

In [0]:
# --- Feature set: just the controls ---
baseline_features = control_cols.copy()

print(f"Baseline features : {len(baseline_features)}")
print(f"Rows              : {len(df)}")

# --- Chronological split: train | 15 val | 15 test ---
n = len(df)
n_test  = 15
n_val   = 15
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy()
val_df   = df.iloc[n_train : n_train + n_val].copy()
test_df  = df.iloc[n_train + n_val :].copy()

print(f"\nTrain: {len(train_df):>3} weeks  "
      f"({train_df['week_date'].min().date()} → {train_df['week_date'].max().date()})")
print(f"Val  : {len(val_df):>3} weeks  "
      f"({val_df['week_date'].min().date()} → {val_df['week_date'].max().date()})")
print(f"Test : {len(test_df):>3} weeks  "
      f"({test_df['week_date'].min().date()} → {test_df['week_date'].max().date()})")

print(f"\nFeatures-to-train-rows: {len(baseline_features) / len(train_df):.2f}")

# --- Scale (fit on TRAIN only, transform all three) ---
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
X_test  = scaler_X.transform(test_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()
y_test  = scaler_y.transform(test_df[[target_col]]).flatten()

print(f"\nX_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")
print(f"y_train  mean: {y_train.mean():+.3f}  std: {y_train.std():.3f}  (expect ~0, ~1)")
print(f"y_val    mean: {y_val.mean():+.3f}  std: {y_val.std():.3f}")
print(f"y_test   mean: {y_test.mean():+.3f}  std: {y_test.std():.3f}")

In [0]:
import pymc as pm
import arviz as az

with pm.Model() as baseline_model:

    # --- Priors ---
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=0.5)

    # --- Linear predictor ---
    mu = intercept + pm.math.dot(X_train, beta)

    # --- Likelihood ---
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    # --- Sample ---
    trace = pm.sample(
        draws=2000,
        tune=2000,
        target_accept=0.9,
        chains=4,
        cores=4,
        random_seed=42,
        return_inferencedata=True,
    )

In [0]:
# --- Convergence diagnostics (fixed for str r_hat) ---
summary = az.summary(trace, var_names=["intercept", "beta", "sigma"])

# Coerce r_hat to float — newer ArviZ returns it as a formatted string
summary["r_hat"] = pd.to_numeric(summary["r_hat"], errors="coerce")

max_rhat    = float(summary["r_hat"].max())
min_ess     = int(summary["ess_bulk"].min())
divergences = int(trace.sample_stats.diverging.sum().item())

print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

# Pass/fail summary
checks = {
    "R-hat":       max_rhat < 1.01,
    "ESS":         min_ess > 400,
    "Divergences": divergences == 0,
}
print("\n" + "─" * 40)
for k, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    print(f"  {k:<12} {mark}")
print("─" * 40)

print("\nWorst 10 by R-hat:")
print(summary.sort_values("r_hat", ascending=False)
              .head(10)[["mean", "sd", "r_hat", "ess_bulk"]])

In [0]:
# --- Generate posterior predictions on train ---
with baseline_model:
    posterior_pred = pm.sample_posterior_predictive(
        trace,
        random_seed=42,
        progressbar=False,
    )

# Mean prediction (scaled space)
pred_train_scaled = (
    posterior_pred.posterior_predictive["y_obs"]
    .mean(dim=["chain", "draw"])
    .values
)

# 95% credible interval (scaled space)
pred_train_lower = (
    posterior_pred.posterior_predictive["y_obs"]
    .quantile(0.025, dim=["chain", "draw"])
    .values
)
pred_train_upper = (
    posterior_pred.posterior_predictive["y_obs"]
    .quantile(0.975, dim=["chain", "draw"])
    .values
)

# --- Inverse-scale back to attendance units ---
pred_train       = scaler_y.inverse_transform(pred_train_scaled.reshape(-1, 1)).flatten()
pred_train_lo    = scaler_y.inverse_transform(pred_train_lower.reshape(-1, 1)).flatten()
pred_train_hi    = scaler_y.inverse_transform(pred_train_upper.reshape(-1, 1)).flatten()
actual_train     = train_df[target_col].values

# --- Metrics ---
from sklearn.metrics import mean_absolute_percentage_error, r2_score

mape_train = mean_absolute_percentage_error(actual_train, pred_train)
r2_train   = r2_score(actual_train, pred_train)

# Coverage: how often does the true value fall inside the 95% CI?
coverage = np.mean((actual_train >= pred_train_lo) & (actual_train <= pred_train_hi))

# Residuals
resid_train = actual_train - pred_train

print(f"Train MAPE     : {mape_train * 100:.2f}%")
print(f"Train R²       : {r2_train:.3f}")
print(f"95% Coverage   : {coverage * 100:.1f}%   (target 90-98%)")
print(f"Residual mean  : {resid_train.mean():,.0f}     (target near 0)")
print(f"Residual std   : {resid_train.std():,.0f}")

In [0]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. Predictions vs actuals with credible interval
ax = axes[0]
ax.plot(train_df["week_date"], actual_train, label="Actual", color="black", linewidth=1.5)
ax.plot(train_df["week_date"], pred_train, label="Predicted (mean)", color="steelblue", linewidth=1.5)
ax.fill_between(train_df["week_date"], pred_train_lo, pred_train_hi,
                alpha=0.2, color="steelblue", label="95% CI")
ax.set_title("Train: actual vs predicted attendance")
ax.set_ylabel("Weekly attendance")
ax.legend()
ax.grid(alpha=0.3)

# 2. Residuals over time
ax = axes[1]
ax.plot(train_df["week_date"], resid_train, color="coral", linewidth=1)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Residuals over time (want: random scatter around 0)")
ax.set_ylabel("Actual − Predicted")
ax.grid(alpha=0.3)

# 3. Residual histogram
ax = axes[2]
ax.hist(resid_train, bins=25, color="coral", edgecolor="black", alpha=0.7)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Residual distribution (want: roughly normal, centered on 0)")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# --- Predict on validation set ---
# Re-build model with X_val data, share posterior from train fit
with pm.Model() as val_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_val.shape[1])
    sigma     = pm.Exponential("sigma", sigma=0.5)

    mu = intercept + pm.math.dot(X_val, beta)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_val)

    val_pred = pm.sample_posterior_predictive(
        trace,
        var_names=["y_obs"],
        predictions=True,
        random_seed=42,
        progressbar=False,
    )

# Mean prediction + 95% CI
pred_val_scaled = val_pred.predictions["y_obs"].mean(dim=["chain", "draw"]).values
pred_val_lo_s   = val_pred.predictions["y_obs"].quantile(0.025, dim=["chain", "draw"]).values
pred_val_hi_s   = val_pred.predictions["y_obs"].quantile(0.975, dim=["chain", "draw"]).values

# Inverse-scale back to attendance
pred_val    = scaler_y.inverse_transform(pred_val_scaled.reshape(-1, 1)).flatten()
pred_val_lo = scaler_y.inverse_transform(pred_val_lo_s.reshape(-1, 1)).flatten()
pred_val_hi = scaler_y.inverse_transform(pred_val_hi_s.reshape(-1, 1)).flatten()
actual_val  = val_df[target_col].values

# --- Metrics ---
mape_val     = mean_absolute_percentage_error(actual_val, pred_val)
r2_val       = r2_score(actual_val, pred_val)
coverage_val = np.mean((actual_val >= pred_val_lo) & (actual_val <= pred_val_hi))
resid_val    = actual_val - pred_val

print(f"VAL MAPE        : {mape_val * 100:.2f}%   (train was 18.11%)")
print(f"VAL R²          : {r2_val:.3f}            (train was 0.703)")
print(f"VAL Coverage    : {coverage_val * 100:.1f}%")
print(f"VAL Residual μ  : {resid_val.mean():,.0f}")
print(f"VAL Residual σ  : {resid_val.std():,.0f}")

# Plot val fit
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(val_df["week_date"], actual_val, "k-", label="Actual", linewidth=1.5)
ax.plot(val_df["week_date"], pred_val, color="steelblue", label="Predicted", linewidth=1.5)
ax.fill_between(val_df["week_date"], pred_val_lo, pred_val_hi, alpha=0.2, color="steelblue", label="95% CI")
ax.set_title(f"Validation set: Aug–Nov 2025 (MAPE {mape_val*100:.1f}%)")
ax.set_ylabel("Weekly attendance")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# --- Compare val plot vs actuals more carefully ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.plot(val_df["week_date"], actual_val, "k-o", label="Actual", linewidth=1.5, markersize=5)
ax.plot(val_df["week_date"], pred_val, color="steelblue", marker="s", label="Predicted", linewidth=1.5, markersize=5)
ax.fill_between(val_df["week_date"], pred_val_lo, pred_val_hi, alpha=0.15, color="steelblue")
ax.set_title("Val: actual vs predicted (week by week)")
ax.set_ylabel("Attendance")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(val_df["week_date"], resid_val, color="coral", width=5)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Val residuals (Actual − Predicted)")
ax.set_ylabel("Residual")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Distribution shift checks: is val outside train range for key features? ---
print("Feature range comparison: train vs val")
print(f"{'Feature':<30} {'Train min':>12} {'Train max':>12} {'Val min':>12} {'Val max':>12} {'Outside?':>10}")
print("─" * 90)

key_features = ["active_films", "headline_cpi", "national_mean_temp"] + \
               [c for c in baseline_features if "_share" in c]

for feat in key_features:
    tr_min, tr_max = train_df[feat].min(), train_df[feat].max()
    v_min, v_max   = val_df[feat].min(), val_df[feat].max()
    outside        = "YES" if (v_min < tr_min or v_max > tr_max) else "no"
    print(f"{feat:<30} {tr_min:>12.2f} {tr_max:>12.2f} {v_min:>12.2f} {v_max:>12.2f} {outside:>10}")

# --- Direction check: does val MAPE drop if we just predict the val mean? ---
naive_pred = np.full_like(actual_val, actual_val.mean(), dtype=float)
naive_mape = mean_absolute_percentage_error(actual_val, naive_pred) * 100
print(f"\nNaive 'predict val mean' MAPE : {naive_mape:.2f}%")
print(f"Model val MAPE                : {mape_val * 100:.2f}%")
print(f"Naive 'predict train mean' MAPE: {mean_absolute_percentage_error(actual_val, np.full_like(actual_val, actual_train.mean(), dtype=float)) * 100:.2f}%")

In [0]:
# Check every feature for val extrapolation
print(f"{'Feature':<32} {'TrMin':>8} {'TrMax':>8} {'VMin':>8} {'VMax':>8} {'Outside':>8}")
print("─" * 80)

outside_features = []
for feat in baseline_features:
    tr_min, tr_max = train_df[feat].min(), train_df[feat].max()
    v_min, v_max   = val_df[feat].min(), val_df[feat].max()
    is_outside     = (v_min < tr_min) or (v_max > tr_max)
    if is_outside:
        outside_features.append(feat)
        marker = "YES"
        print(f"{feat:<32} {tr_min:>8.2f} {tr_max:>8.2f} {v_min:>8.2f} {v_max:>8.2f} {marker:>8}")

print(f"\nTotal features extrapolating in val: {len(outside_features)}")

# Also check coefficient magnitudes — are some betas suspiciously large?
beta_means = summary.loc[summary.index.str.startswith("beta"), "mean"].astype(float).values
beta_sds   = summary.loc[summary.index.str.startswith("beta"), "sd"].astype(float).values

beta_df = pd.DataFrame({
    "feature": baseline_features,
    "beta_mean": beta_means,
    "beta_sd": beta_sds,
    "abs_mean": np.abs(beta_means),
})
print("\nTop 10 largest beta magnitudes (potential overfitters):")
print(beta_df.sort_values("abs_mean", ascending=False).head(10).to_string(index=False))

In [0]:
# --- Build leaner feature set ---

drop_features = [
    "national_max_temp",       # Keep national_mean_temp, drop max + min
    "national_min_temp",
    "national_temp_anomaly",   # Highly correlated with the temp trio
    "action_intensity",        # Drop ALL genre intensities — let cat1-5_share carry it
    "adaptation_intensity",
    "adventure_intensity",
    "animated_intensity",
    "biopic_intensity",
    "comedy_intensity",
    "crime_intensity",
    "documentary_intensity",
    "drama_intensity",
    "family_intensity",
    "fantasy_intensity",
    "foreign_intensity",
    "horror_intensity",
    "musical_intensity",
    "mystery_intensity",
    "period_intensity",
    "political_intensity",
    "romance_intensity",
    "romanticcomedy_intensity",
    "scifi_intensity",
    "sequel_intensity",
    "suspense_intensity",
    "thriller_intensity",
]

baseline_features_trim = [f for f in baseline_features if f not in drop_features]

print(f"Original features : {len(baseline_features)}")
print(f"Dropped           : {len([f for f in drop_features if f in baseline_features])}")
print(f"New feature count : {len(baseline_features_trim)}")
print(f"Features-to-train-rows: {len(baseline_features_trim) / len(train_df):.2f}")
print(f"\nRemaining features:")
for f in baseline_features_trim:
    print(f"  {f}")

In [0]:
baseline_features = baseline_features_trim   # overwrite to keep naming consistent

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
X_test  = scaler_X.transform(test_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()
y_test  = scaler_y.transform(test_df[[target_col]]).flatten()

print(f"X_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")

In [0]:
summary = az.summary(trace, var_names=["intercept", "beta", "sigma"])
summary["r_hat"] = pd.to_numeric(summary["r_hat"], errors="coerce")

max_rhat    = float(summary["r_hat"].max())
min_ess     = int(summary["ess_bulk"].min())
divergences = int(trace.sample_stats.diverging.sum().item())

print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

In [0]:
with baseline_model:
    pp_train = pm.sample_posterior_predictive(trace, random_seed=42, progressbar=False)

pred_train_s  = pp_train.posterior_predictive["y_obs"].mean(dim=["chain","draw"]).values
pred_train_lo = pp_train.posterior_predictive["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_train_hi = pp_train.posterior_predictive["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_train       = scaler_y.inverse_transform(pred_train_s.reshape(-1,1)).flatten()
pred_train_lo_us = scaler_y.inverse_transform(pred_train_lo.reshape(-1,1)).flatten()
pred_train_hi_us = scaler_y.inverse_transform(pred_train_hi.reshape(-1,1)).flatten()
actual_train     = train_df[target_col].values

mape_train     = mean_absolute_percentage_error(actual_train, pred_train) * 100
r2_train       = r2_score(actual_train, pred_train)
coverage_train = np.mean((actual_train >= pred_train_lo_us) & (actual_train <= pred_train_hi_us)) * 100

print(f"TRAIN MAPE     : {mape_train:.2f}%   (was 18.11% with 45 feats)")
print(f"TRAIN R²       : {r2_train:.3f}      (was 0.703)")
print(f"TRAIN Coverage : {coverage_train:.1f}%  (target 90-98%)")

In [0]:
with pm.Model() as val_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_val.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)
    mu_v      = intercept + pm.math.dot(X_val, beta)
    y_obs_v   = pm.Normal("y_obs", mu=mu_v, sigma=sigma, observed=y_val)
    pp_val    = pm.sample_posterior_predictive(
        trace, var_names=["y_obs"], predictions=True,
        random_seed=42, progressbar=False,
    )

pred_val_s  = pp_val.predictions["y_obs"].mean(dim=["chain","draw"]).values
pred_val_lo = pp_val.predictions["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_val_hi = pp_val.predictions["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_val       = scaler_y.inverse_transform(pred_val_s.reshape(-1,1)).flatten()
pred_val_lo_us = scaler_y.inverse_transform(pred_val_lo.reshape(-1,1)).flatten()
pred_val_hi_us = scaler_y.inverse_transform(pred_val_hi.reshape(-1,1)).flatten()
actual_val     = val_df[target_col].values

mape_val     = mean_absolute_percentage_error(actual_val, pred_val) * 100
r2_val       = r2_score(actual_val, pred_val)
coverage_val = np.mean((actual_val >= pred_val_lo_us) & (actual_val <= pred_val_hi_us)) * 100

print(f"VAL MAPE     : {mape_val:.2f}%   (was 49.25% with 45 feats)")
print(f"VAL R²       : {r2_val:.3f}      (was -2.173)")
print(f"VAL Coverage : {coverage_val:.1f}%")

# Plus the diagnostic — has active_films flipped to positive?
print("\nKey coefficient sanity check:")
beta_means_new = summary.loc[summary.index.str.startswith("beta"), "mean"].astype(float).values
for f, b in zip(baseline_features, beta_means_new):
    if f in ["active_films", "headline_cpi", "national_mean_temp", "bad_weather_flag", "is_long_weekend"]:
        print(f"  {f:<25}  β = {b:+.3f}")

In [0]:
import pymc as pm
import arviz as az

# --- Step 1: trim feature list and rebuild matrices ---

drop_features = [
    "national_max_temp", "national_min_temp", "national_temp_anomaly",
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity",
]

baseline_features = [f for f in control_cols if f not in drop_features]
print(f"Feature count: {len(baseline_features)}")

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
X_test  = scaler_X.transform(test_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()
y_test  = scaler_y.transform(test_df[[target_col]]).flatten()

print(f"X_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")
assert X_train.shape[1] == 19, f"Expected 19 features, got {X_train.shape[1]}"

In [0]:
# --- Step 2: refit baseline model with trimmed features ---

with pm.Model() as baseline_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train, beta)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    trace = pm.sample(
        draws=2000, tune=2000,
        target_accept=0.9,
        chains=4, cores=4,
        random_seed=42,
        return_inferencedata=True,
    )

print(f"Posterior beta shape: {trace.posterior['beta'].shape}")  # should end in (..., 19)

In [0]:
summary = az.summary(trace, var_names=["intercept", "beta", "sigma"])
summary["r_hat"] = pd.to_numeric(summary["r_hat"], errors="coerce")

max_rhat    = float(summary["r_hat"].max())
min_ess     = int(summary["ess_bulk"].min())
divergences = int(trace.sample_stats.diverging.sum().item())

print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

In [0]:
with baseline_model:
    pp_train = pm.sample_posterior_predictive(trace, random_seed=42, progressbar=False)

pred_train_s  = pp_train.posterior_predictive["y_obs"].mean(dim=["chain","draw"]).values
pred_train_lo = pp_train.posterior_predictive["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_train_hi = pp_train.posterior_predictive["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_train       = scaler_y.inverse_transform(pred_train_s.reshape(-1,1)).flatten()
pred_train_lo_us = scaler_y.inverse_transform(pred_train_lo.reshape(-1,1)).flatten()
pred_train_hi_us = scaler_y.inverse_transform(pred_train_hi.reshape(-1,1)).flatten()
actual_train     = train_df[target_col].values

mape_train     = mean_absolute_percentage_error(actual_train, pred_train) * 100
r2_train       = r2_score(actual_train, pred_train)
coverage_train = np.mean((actual_train >= pred_train_lo_us) & (actual_train <= pred_train_hi_us)) * 100

print(f"TRAIN MAPE     : {mape_train:.2f}%   (was 18.11% with 45 feats)")
print(f"TRAIN R²       : {r2_train:.3f}      (was 0.703)")
print(f"TRAIN Coverage : {coverage_train:.1f}%  (target 90-98%)")

In [0]:
with pm.Model() as val_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_val.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)
    mu_v      = intercept + pm.math.dot(X_val, beta)
    y_obs_v   = pm.Normal("y_obs", mu=mu_v, sigma=sigma, observed=y_val)
    pp_val    = pm.sample_posterior_predictive(
        trace, var_names=["y_obs"], predictions=True,
        random_seed=42, progressbar=False,
    )

pred_val_s  = pp_val.predictions["y_obs"].mean(dim=["chain","draw"]).values
pred_val_lo = pp_val.predictions["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_val_hi = pp_val.predictions["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_val       = scaler_y.inverse_transform(pred_val_s.reshape(-1,1)).flatten()
pred_val_lo_us = scaler_y.inverse_transform(pred_val_lo.reshape(-1,1)).flatten()
pred_val_hi_us = scaler_y.inverse_transform(pred_val_hi.reshape(-1,1)).flatten()
actual_val     = val_df[target_col].values

mape_val     = mean_absolute_percentage_error(actual_val, pred_val) * 100
r2_val       = r2_score(actual_val, pred_val)
coverage_val = np.mean((actual_val >= pred_val_lo_us) & (actual_val <= pred_val_hi_us)) * 100

print(f"VAL MAPE     : {mape_val:.2f}%   (was 49.25% with 45 feats)")
print(f"VAL R²       : {r2_val:.3f}      (was -2.173)")
print(f"VAL Coverage : {coverage_val:.1f}%")

# Plus the diagnostic — has active_films flipped to positive?
print("\nKey coefficient sanity check:")
beta_means_new = summary.loc[summary.index.str.startswith("beta"), "mean"].astype(float).values
for f, b in zip(baseline_features, beta_means_new):
    if f in ["active_films", "headline_cpi", "national_mean_temp", "bad_weather_flag", "is_long_weekend"]:
        print(f"  {f:<25}  β = {b:+.3f}")

In [0]:
# Is active_films correlated with time (date)?
train_df_temp = train_df.copy()
train_df_temp["week_idx"] = np.arange(len(train_df_temp))

print("Correlations with week index (i.e., time):")
for f in ["active_films", "headline_cpi", "national_mean_temp", target_col]:
    r = train_df_temp[f].corr(train_df_temp["week_idx"])
    print(f"  {f:<25} {r:+.3f}")

print("\nYearly means:")
print(train_df_temp.groupby(train_df_temp["week_date"].dt.year)[
    ["active_films", target_col, "headline_cpi"]
].mean().round(0))

In [0]:
# What does active_films correlate with in the training data?
corr_with_active = train_df[baseline_features + [target_col]].corr()["active_films"].sort_values()
print("active_films correlations with other features:")
print(corr_with_active.round(3))

In [0]:
# Show top 10 attendance weeks and what features looked like

top_weeks = train_df.nlargest(10, target_col)[

    ["week_date", target_col, "active_films",

     "cat1_share", "cat2_share", "cat3_share", "cat4_share", "cat5_share"]

]

print("\nTop 10 attendance weeks in train:")

print(top_weeks.to_string(index=False))

print("\nBottom 10 attendance weeks in train:")

bot_weeks = train_df.nsmallest(10, target_col)[

    ["week_date", target_col, "active_films",

     "cat1_share", "cat2_share", "cat3_share", "cat4_share", "cat5_share"]

]

print(bot_weeks.to_string(index=False))